In [8]:
import numpy as np

np.set_printoptions(suppress=True, linewidth=160)
np.seterr(invalid="ignore", divide="ignore")

DATA_PATH = "../data/titanic.csv"


In [9]:

# Exercise 1.1: Load the dataset

COLUMN_NAMES = np.genfromtxt(
    DATA_PATH,
    delimiter=",",
    dtype=str,
    max_rows=1,
    encoding="utf-8"
)

NUMPY_TITANIC = np.genfromtxt(
    DATA_PATH,
    delimiter=",",
    dtype=str,
    skip_header=1,
    encoding="utf-8",
    autostrip=True,
    filling_values=""
)

print("Rows:", NUMPY_TITANIC.shape[0])
print("Columns:", NUMPY_TITANIC.shape[1])
print(NUMPY_TITANIC[:10])

missing_mask = (NUMPY_TITANIC == "")

for j, name in enumerate(COLUMN_NAMES):
    count = np.sum(missing_mask[:, j])
    percentage = count / NUMPY_TITANIC.shape[0] * 100
    print(f"{name}: {count} missing ({percentage:.4f}%)")


Rows: 891
Columns: 15
[['0' '3' 'male' '22.0' '1' '0' '7.25' 'S' 'Third' 'man' 'True' '' 'Southampton' 'no' 'False']
 ['1' '1' 'female' '38.0' '1' '0' '71.2833' 'C' 'First' 'woman' 'False' 'C' 'Cherbourg' 'yes' 'False']
 ['1' '3' 'female' '26.0' '0' '0' '7.925' 'S' 'Third' 'woman' 'False' '' 'Southampton' 'yes' 'True']
 ['1' '1' 'female' '35.0' '1' '0' '53.1' 'S' 'First' 'woman' 'False' 'C' 'Southampton' 'yes' 'False']
 ['0' '3' 'male' '35.0' '0' '0' '8.05' 'S' 'Third' 'man' 'True' '' 'Southampton' 'no' 'True']
 ['0' '3' 'male' '' '0' '0' '8.4583' 'Q' 'Third' 'man' 'True' '' 'Queenstown' 'no' 'True']
 ['0' '1' 'male' '54.0' '0' '0' '51.8625' 'S' 'First' 'man' 'True' 'E' 'Southampton' 'no' 'True']
 ['0' '3' 'male' '2.0' '3' '1' '21.075' 'S' 'Third' 'child' 'False' '' 'Southampton' 'no' 'False']
 ['1' '3' 'female' '27.0' '0' '2' '11.1333' 'S' 'Third' 'woman' 'False' '' 'Southampton' 'yes' 'False']
 ['1' '2' 'female' '14.0' '1' '0' '30.0708' 'C' 'Second' 'child' 'False' '' 'Cherbourg' 'ye

In [10]:

# Exercise 1.2: Remove redundant and high-missing columns

redundant_columns = ["class", "who", "embark_town", "alive"]

redundant_indices = [
    np.where(COLUMN_NAMES == name)[0][0]
    for name in redundant_columns
]

NUMPY_TITANIC = np.delete(
    NUMPY_TITANIC, redundant_indices, axis=1
)
COLUMN_NAMES = np.delete(COLUMN_NAMES, redundant_indices)

missing_mask = (NUMPY_TITANIC == "")
missing_counts = np.sum(missing_mask, axis=0)

high_missing_mask = (
    missing_counts / NUMPY_TITANIC.shape[0] > 0.5
)

high_missing_columns = COLUMN_NAMES[high_missing_mask]

NUMPY_TITANIC = NUMPY_TITANIC[:, ~high_missing_mask]
COLUMN_NAMES = COLUMN_NAMES[~high_missing_mask]

print("Redundant columns removed:", redundant_columns)
print("Columns with more than 50% missing removed:",
      high_missing_columns.tolist())
print("Remaining columns:", COLUMN_NAMES.tolist())


Redundant columns removed: ['class', 'who', 'embark_town', 'alive']
Columns with more than 50% missing removed: ['deck']
Remaining columns: ['survived', 'pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked', 'adult_male', 'alone']


In [11]:

# Exercise 1.3a

TARGET_COLUMN = "survived"

NUMERICAL_NAMES = [
    "pclass", "age", "sibsp", "parch", "fare"
]

CATEGORICAL_NAMES = [
    name for name in COLUMN_NAMES
    if name not in NUMERICAL_NAMES + [TARGET_COLUMN]
]

print("Target column:", TARGET_COLUMN)
print("Numerical features:", NUMERICAL_NAMES)
print("Categorical features:", CATEGORICAL_NAMES)


Target column: survived
Numerical features: ['pclass', 'age', 'sibsp', 'parch', 'fare']
Categorical features: [np.str_('sex'), np.str_('embarked'), np.str_('adult_male'), np.str_('alone')]


In [16]:

NUMERICAL_FEATURES = np.full(
    (NUMPY_TITANIC.shape[0], len(NUMERICAL_NAMES)),
    np.nan,
    dtype=float
)

for j, name in enumerate(NUMERICAL_NAMES):
    idx = np.where(COLUMN_NAMES == name)[0][0]
    column = NUMPY_TITANIC[:, idx]
    valid = column != ""
    NUMERICAL_FEATURES[valid, j] = column[valid].astype(float)

print("Data type of NUMERICAL_FEATURES:",
      NUMERICAL_FEATURES.dtype)

print("Missing values per numerical feature:", {
    name: int(np.isnan(NUMERICAL_FEATURES[:, j]).sum())
    for j, name in enumerate(NUMERICAL_NAMES)
})


Data type of NUMERICAL_FEATURES: float64
Missing values per numerical feature: {'pclass': 0, 'age': 177, 'sibsp': 0, 'parch': 0, 'fare': 0}


In [17]:

# Exercise 1.4

categorical_indices = [
    np.where(COLUMN_NAMES == name)[0][0]
    for name in CATEGORICAL_NAMES
]

CATEGORICAL_FEATURES = NUMPY_TITANIC[
    :, categorical_indices
].copy()

for j, name in enumerate(CATEGORICAL_NAMES):
    values, counts = np.unique(
        CATEGORICAL_FEATURES[:, j],
        return_counts=True
    )
    print(name + ":", dict(zip(values.tolist(), counts.tolist())))


sex: {'female': 314, 'male': 577}
embarked: {'': 2, 'C': 168, 'Q': 77, 'S': 644}
adult_male: {'False': 354, 'True': 537}
alone: {'False': 354, 'True': 537}


In [18]:

# Exercise 2: Helper for numerical modes

def column_modes(array, ignore_nan=False):
    modes = []

    for j in range(array.shape[1]):
        column = array[:, j]

        if ignore_nan:
            column = column[~np.isnan(column)]

        if len(column) == 0:
            modes.append(np.nan)
            continue

        values, counts = np.unique(column, return_counts=True)
        modes.append(values[counts == counts.max()].min())

    return np.array(modes, dtype=float)


# 2.1 Without excluding missing elements

print("Without excluding missing elements:")
print("Mean:  ", np.mean(NUMERICAL_FEATURES, axis=0))
print("Median:", np.median(NUMERICAL_FEATURES, axis=0))
print("Mode:  ", column_modes(NUMERICAL_FEATURES))
print("Std:   ", np.std(NUMERICAL_FEATURES, axis=0))
print("Min:   ", np.min(NUMERICAL_FEATURES, axis=0))
print("Max:   ", np.max(NUMERICAL_FEATURES, axis=0))


# 2.2 Excluding missing elements

print("\nExcluding missing elements:")
print("Mean:  ", np.nanmean(NUMERICAL_FEATURES, axis=0))
print("Median:", np.nanmedian(NUMERICAL_FEATURES, axis=0))
print("Mode:  ", column_modes(NUMERICAL_FEATURES, ignore_nan=True))
print("Std:   ", np.nanstd(NUMERICAL_FEATURES, axis=0))
print("Min:   ", np.nanmin(NUMERICAL_FEATURES, axis=0))
print("Max:   ", np.nanmax(NUMERICAL_FEATURES, axis=0))


Without excluding missing elements:
Mean:   [ 2.30864198         nan  0.52300786  0.38159371 32.20420797]
Median: [ 3.         nan  0.      0.     14.4542]
Mode:   [3.    nan 0.   0.   8.05]
Std:    [ 0.83560193         nan  1.10212444  0.80560476 49.66553444]
Min:    [ 1. nan  0.  0.  0.]
Max:    [  3.          nan   8.       6.     512.3292]

Excluding missing elements:
Mean:   [ 2.30864198 29.69911765  0.52300786  0.38159371 32.20420797]
Median: [ 3.     28.      0.      0.     14.4542]
Mode:   [ 3.   24.    0.    0.    8.05]
Std:    [ 0.83560193 14.51632115  1.10212444  0.80560476 49.66553444]
Min:    [1.   0.42 0.   0.   0.  ]
Max:    [  3.      80.       8.       6.     512.3292]


In [19]:

def print_rounded(label, values):
    print(label, np.array2string(
        values, precision=4, floatmode="fixed"
    ))


In [20]:

target_idx = np.where(COLUMN_NAMES == TARGET_COLUMN)[0][0]

TARGET_VALUES = NUMPY_TITANIC[:, target_idx].astype(float)

print(f"Mean of target column: {np.mean(TARGET_VALUES):.4f}")


Mean of target column: 0.3838


In [21]:

# Exercise 3.1

for j, name in enumerate(CATEGORICAL_NAMES):
    column = CATEGORICAL_FEATURES[:, j]
    missing = column == ""

    if np.any(missing):
        observed = column[~missing]
        values, counts = np.unique(observed, return_counts=True)

        # np.unique sorts values; ties choose the first, hence smallest
        fill_value = values[counts == counts.max()][0]
        count_missing = int(missing.sum())

        CATEGORICAL_FEATURES[missing, j] = fill_value

        print(
            f"{name}: {count_missing} missing replaced "
            f"with {fill_value}"
        )

print(
    "Missing values remaining in categorical features:",
    int(np.sum(CATEGORICAL_FEATURES == ""))
)


embarked: 2 missing replaced with S
Missing values remaining in categorical features: 0


In [22]:

# Exercise 3.2

ordinal_columns = []

for j, name in enumerate(CATEGORICAL_NAMES):
    categories, inverse = np.unique(
        CATEGORICAL_FEATURES[:, j],
        return_inverse=True
    )

    ordinal_columns.append(inverse)

    mapping = {
        category: int(code)
        for code, category in enumerate(categories)
    }

    print(name + ":", mapping)

ORDINAL_ENCODED = np.column_stack(ordinal_columns)

print("Ordinal encoded (first 5 rows):")
print(ORDINAL_ENCODED[:5])


sex: {np.str_('female'): 0, np.str_('male'): 1}
embarked: {np.str_('C'): 0, np.str_('Q'): 1, np.str_('S'): 2}
adult_male: {np.str_('False'): 0, np.str_('True'): 1}
alone: {np.str_('False'): 0, np.str_('True'): 1}
Ordinal encoded (first 5 rows):
[[1 2 1 0]
 [0 0 0 0]
 [0 2 0 1]
 [0 2 0 0]
 [1 2 1 1]]


In [23]:

# Exercise 3.3

onehot_columns = []
onehot_names = []

for j, name in enumerate(CATEGORICAL_NAMES):
    categories, inverse = np.unique(
        CATEGORICAL_FEATURES[:, j],
        return_inverse=True
    )

    encoded = np.eye(len(categories), dtype=int)[inverse]
    onehot_columns.append(encoded)

    onehot_names.extend([
        f"{name}_{category}" for category in categories
    ])

ONEHOT_ENCODED = np.hstack(onehot_columns)

print("One-hot encoded (first 5 rows):")
print(ONEHOT_ENCODED[:5])


One-hot encoded (first 5 rows):
[[0 1 0 0 1 0 1 1 0]
 [1 0 1 0 0 1 0 1 0]
 [1 0 0 0 1 1 0 0 1]
 [1 0 0 0 1 1 0 1 0]
 [0 1 0 0 1 0 1 0 1]]


In [24]:

# Exercise 3.4

NUMERICAL_ORDINAL = np.hstack([
    NUMERICAL_FEATURES,
    ORDINAL_ENCODED
])

NUMERICAL_ONEHOT = np.hstack([
    NUMERICAL_FEATURES,
    ONEHOT_ENCODED
])

print("Shape of NUMERICAL_ORDINAL:", NUMERICAL_ORDINAL.shape)
print("Shape of NUMERICAL_ONEHOT:", NUMERICAL_ONEHOT.shape)


Shape of NUMERICAL_ORDINAL: (891, 9)
Shape of NUMERICAL_ONEHOT: (891, 14)


In [25]:

# Exercise 4.1: Mean imputation

mean_fill_values = np.nanmean(NUMERICAL_FEATURES, axis=0)

MEAN_IMPUTED = NUMERICAL_FEATURES.copy()

for j in range(MEAN_IMPUTED.shape[1]):
    missing = np.isnan(MEAN_IMPUTED[:, j])
    MEAN_IMPUTED[missing, j] = mean_fill_values[j]

print("Mean imputation fill values:",
      np.round(mean_fill_values, 4))


# Exercise 4.2: Median imputation

median_fill_values = np.nanmedian(NUMERICAL_FEATURES, axis=0)

MEDIAN_IMPUTED = NUMERICAL_FEATURES.copy()

for j in range(MEDIAN_IMPUTED.shape[1]):
    missing = np.isnan(MEDIAN_IMPUTED[:, j])
    MEDIAN_IMPUTED[missing, j] = median_fill_values[j]

print("Median imputation fill values:",
      np.round(median_fill_values, 4))


Mean imputation fill values: [ 2.3086 29.6991  0.523   0.3816 32.2042]
Median imputation fill values: [ 3.     28.      0.      0.     14.4542]


In [26]:

# Exercise 4.3

for j, name in enumerate(NUMERICAL_NAMES):
    if np.isnan(NUMERICAL_FEATURES[:, j]).any():
        original = NUMERICAL_FEATURES[:, j]
        mean_col = MEAN_IMPUTED[:, j]
        median_col = MEDIAN_IMPUTED[:, j]

        print(name + ":")
        print(
            f"  NUMERICAL_FEATURES: mean {np.nanmean(original):.4f}, "
            f"std {np.nanstd(original):.4f}"
        )
        print(
            f"  MEAN_IMPUTED:       mean {np.mean(mean_col):.4f}, "
            f"std {np.std(mean_col):.4f}"
        )
        print(
            f"  MEDIAN_IMPUTED:     mean {np.mean(median_col):.4f}, "
            f"std {np.std(median_col):.4f}"
        )


age:
  NUMERICAL_FEATURES: mean 29.6991, std 14.5163
  MEAN_IMPUTED:       mean 29.6991, std 12.9947
  MEDIAN_IMPUTED:     mean 29.3616, std 13.0124


In [27]:

# Exercise 5.1

col_min = np.min(MEDIAN_IMPUTED, axis=0)
col_max = np.max(MEDIAN_IMPUTED, axis=0)

MINMAX_NORMALISED = (
    (MEDIAN_IMPUTED - col_min)
    / (col_max - col_min)
)

print("MINMAX_NORMALISED (first 5 rows):")
print(np.round(MINMAX_NORMALISED[:5], 4))


MINMAX_NORMALISED (first 5 rows):
[[1.     0.2712 0.125  0.     0.0142]
 [0.     0.4722 0.125  0.     0.1391]
 [1.     0.3214 0.     0.     0.0155]
 [0.     0.4345 0.125  0.     0.1036]
 [1.     0.4345 0.     0.     0.0157]]


In [28]:

# Exercise 5.2

col_mean = np.mean(MEDIAN_IMPUTED, axis=0)
col_std = np.std(MEDIAN_IMPUTED, axis=0)

ZSCORE_STANDARDISED = (
    (MEDIAN_IMPUTED - col_mean) / col_std
)

print("ZSCORE_STANDARDISED (first 5 rows):")
print(np.round(ZSCORE_STANDARDISED[:5], 4))


ZSCORE_STANDARDISED (first 5 rows):
[[ 0.8274 -0.5657  0.4328 -0.4737 -0.5024]
 [-1.5661  0.6639  0.4328 -0.4737  0.7868]
 [ 0.8274 -0.2583 -0.4745 -0.4737 -0.4889]
 [-1.5661  0.4333  0.4328 -0.4737  0.4207]
 [ 0.8274  0.4333 -0.4745 -0.4737 -0.4863]]
